# Orientation des composants avec YOLO-OBB (Module 3)

Ce notebook est adapté au dataset **`wiring harness connectors segmen.v1i.yolov8-obb`**.

### Ce que fait ce notebook
1. Vérifie et charge automatiquement le dataset OBB (`data.yaml` et labels au format OBB).
2. Audite et nettoie les fichiers de labels (`.txt`) pour garantir la compatibilité avec la détection orientée (`yolo11n-obb.pt` : classe + 8 coordonnées normalisées dans `[0, 1]`), en éliminant les lignes corrompues ou hors bornes (comme celles signalées lors de l'entraînement) et en régénérant le cache Ultralytics (`*.cache`).
3. Entraîne **YOLO-OBB** (`yolo11n-obb.pt`) par **paliers de 10 époques** (jusqu'à 100 époques au total) en stockant chaque modèle intermédiaire (`best_ep10.pt`, `best_ep20.pt`, ..., `best_ep100.pt`) dans `CHECKPOINTS_DIR`.

## 1. Importation des bibliothèques

In [1]:
import shutil
import zipfile
from collections import Counter
from pathlib import Path
import yaml
import torch
from ultralytics import YOLO

## 2. Extraction (si nécessaire) et vérification du dataset OBB

In [2]:
DATASET_DIR = Path(r"D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb").resolve()
ZIP_PATH = Path(r"D:\S5\Projet_d_expertise\projet\data\raw\wiring harness connectors segmen.v1i.yolov8-obb.zip").resolve()
DATA_YAML = DATASET_DIR / "data.yaml"

if not DATASET_DIR.exists() and ZIP_PATH.exists():
    print(f"Extraction de {ZIP_PATH} vers {DATASET_DIR}...")
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATASET_DIR)

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Fichier data.yaml introuvable : {DATA_YAML}\nVérifiez le chemin de DATASET_DIR ou ZIP_PATH."
    )

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Device utilisé : {DEVICE}")
print(f"Dataset prêt : {DATASET_DIR}")
print(f"Fichier trouvé : {DATA_YAML}")
print("Contenu de data.yaml :")
print(yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True))

Device utilisé : cpu
Dataset prêt : D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb
Fichier trouvé : D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml
Contenu de data.yaml :
train: train/images
val: valid/images
test: test/images
names:
  0: cable
  1: cable-tie
  2: clip
  3: connector
  4: label



## 3. Audit et nettoyage des annotations pour l'orientation (`yolo_obb`)

In [3]:
CLEAN_INVALID_OBB = True

for split in ["train", "valid", "test"]:
    lbl_dir = DATASET_DIR / split / "labels"
    if not lbl_dir.exists():
        continue

    stats = Counter()
    for txt_file in lbl_dir.glob("*.txt"):
        lines = [ln.strip() for ln in txt_file.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if not lines:
            stats["empty_bg_files"] += 1
            continue

        valid_lines = []
        for ln in lines:
            parts = ln.split()
            # Format OBB : class_id + 8 coordonnées normalisées (x1 y1 x2 y2 x3 y3 x4 y4)
            if len(parts) == 9:
                try:
                    coords = [float(v) for v in parts[1:]]
                    if all(0.0 <= c <= 1.0 for c in coords):
                        stats["obb_boxes"] += 1
                        valid_lines.append(ln)
                    else:
                        stats["out_of_bounds"] += 1
                        if not CLEAN_INVALID_OBB:
                            valid_lines.append(ln)
                except ValueError:
                    stats["invalid"] += 1
            else:
                stats["invalid"] += 1

        if CLEAN_INVALID_OBB and len(valid_lines) != len(lines):
            txt_file.write_text("\n".join(valid_lines) + ("\n" if valid_lines else ""), encoding="utf-8")

    for cache_file in (DATASET_DIR / split).glob("*.cache"):
        cache_file.unlink(missing_ok=True)

    print(f"[{split}] {dict(stats)}")

[train] {'obb_boxes': 872, 'empty_bg_files': 12, 'out_of_bounds': 123}
[valid] {'obb_boxes': 88, 'out_of_bounds': 18, 'empty_bg_files': 2}
[test] {'obb_boxes': 20, 'out_of_bounds': 5}


## 4. Chargement du modèle pré-entraîné (`yolo11n-obb.pt`) et préparation du stockage par paliers

In [4]:
MODELS_DIR = Path(r"D:\S5\Projet_d_expertise\projet\models").resolve()
CHECKPOINTS_DIR = MODELS_DIR / "03_orientation_des_composants_paliers"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_WEIGHTS = "yolo11n-obb.pt"
model = YOLO(BASE_WEIGHTS)

def train_10_epochs(weights_source, run_name: str, saved_model_name: str) -> Path:
    """Entraîne 10 époques OBB à partir de `weights_source` et stocke le meilleur modèle du palier."""
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    stage_model = YOLO(str(weights_source))
    stage_model.train(
        data=str(DATA_YAML),
        task="obb",
        epochs=10,
        imgsz=640,
        batch=16,
        patience=5,
        workers=0,
        device=DEVICE,
        project=str(MODELS_DIR),
        name=run_name,
        exist_ok=True,
        save=True,
        plots=True,
    )
    trainer_best = Path(stage_model.trainer.best)
    trainer_last = Path(stage_model.trainer.last)
    src_pt = trainer_best if trainer_best.exists() else trainer_last
    stored_path = CHECKPOINTS_DIR / saved_model_name
    stored_path.parent.mkdir(parents=True, exist_ok=True)
    if not src_pt.exists():
        raise FileNotFoundError(f"Fichier de poids introuvable : {src_pt}")
    shutil.copy2(src_pt, stored_path)
    print(f"Modèle stocké : {stored_path}")
    return stored_path

## 5. Entraînement par paliers de 10 époques (100 époques au total)

### Palier (1 à 10 époques)

In [5]:
model_ep10 = train_10_epochs(
    weights_source=BASE_WEIGHTS,
    run_name="03_orientation_des_composants/ep01_10",
    saved_model_name="best_ep10.pt",
)
model = YOLO(str(model_ep10))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model

In [6]:
# Cellule de reprise (si vous avez déjà entraîné les 10 premières époques)
CHECKPOINTS_DIR = Path(r"D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants_paliers")
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

best_ep10_src = Path(r"D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants\ep01_10\weights\best.pt")
if not best_ep10_src.exists():
    # Fallback vers l'ancien dossier d'exécution de 10 époques s'il existe
    best_ep10_src = Path(r"D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants\weights\best.pt")

model_ep10 = CHECKPOINTS_DIR / "best_ep10.pt"
if best_ep10_src.exists() and not model_ep10.exists():
    shutil.copy2(best_ep10_src, model_ep10)
if model_ep10.exists():
    model = YOLO(str(model_ep10))
    print(f"Reprise prête depuis : {model_ep10}")

Reprise prête depuis : D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants_paliers\best_ep10.pt


### Palier (11 à 20 époques)

In [7]:
model_ep20 = train_10_epochs(
    weights_source=model_ep10,
    run_name="03_orientation_des_composants/ep11_20",
    saved_model_name="best_ep20.pt",
)
model = YOLO(str(model_ep20))

Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants_paliers\best_ep10.pt, momentum=

### Palier (21 à 30 époques)

In [8]:
model_ep30 = train_10_epochs(
    weights_source=model_ep20,
    run_name="03_orientation_des_composants/ep21_30",
    saved_model_name="best_ep30.pt",
)
model = YOLO(str(model_ep30))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model

### Palier (31 à 40 époques)

In [9]:
model_ep40 = train_10_epochs(
    weights_source=model_ep30,
    run_name="03_orientation_des_composants/ep31_40",
    saved_model_name="best_ep40.pt",
)
model = YOLO(str(model_ep40))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model

### Palier (41 à 50 époques)

In [10]:
model_ep50 = train_10_epochs(
    weights_source=model_ep40,
    run_name="03_orientation_des_composants/ep41_50",
    saved_model_name="best_ep50.pt",
)
model = YOLO(str(model_ep50))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model

### Palier (51 à 60 époques)

In [11]:
model_ep60 = train_10_epochs(
    weights_source=model_ep50,
    run_name="03_orientation_des_composants/ep51_60",
    saved_model_name="best_ep60.pt",
)
model = YOLO(str(model_ep60))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model

### Palier (61 à 70 époques)

In [12]:
model_ep70 = train_10_epochs(
    weights_source=model_ep60,
    run_name="03_orientation_des_composants/ep61_70",
    saved_model_name="best_ep70.pt",
)
model = YOLO(str(model_ep70))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model

### Palier (71 à 80 époques)

In [13]:
model_ep80 = train_10_epochs(
    weights_source=model_ep70,
    run_name="03_orientation_des_composants/ep71_80",
    saved_model_name="best_ep80.pt",
)
model = YOLO(str(model_ep80))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model

### Palier (81 à 90 époques)

In [14]:
model_ep90 = train_10_epochs(
    weights_source=model_ep80,
    run_name="03_orientation_des_composants/ep81_90",
    saved_model_name="best_ep90.pt",
)
model = YOLO(str(model_ep90))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model

### Palier (91 à 100 époques)

In [15]:
model_ep100 = train_10_epochs(
    weights_source=model_ep90,
    run_name="03_orientation_des_composants/ep91_100",
    saved_model_name="best_ep100.pt",
)
model = YOLO(str(model_ep100))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model